# 12.4: Workflow Orchestration, Satu Sistem Tiga Jalur

**Yang akan kita bangun**

Satu graph yang menerima pertanyaan apa pun, memutuskan sendiri cabang mana yang
tepat, lalu menjalankan cabang itu:

```
                        ┌──► rag ──────┐
  input ──► deteksi_intent ──► alat ───┼──► jawab ──► jawaban akhir
                        └──► langsung ─┘
```

Tiga cabang, satu titik temu. Titik temu itu penting: berapa pun cabang yang
ada, sistem tetap mengeluarkan satu jawaban dengan bentuk yang sama.

In [ ]:
%pip install -q langgraph

In [ ]:
import os
import sys
import textwrap
from typing_extensions import TypedDict

sys.path.insert(0, "..")

from dotenv import load_dotenv
from google import genai
from google.genai import types
from langgraph.graph import StateGraph, START, END

from rag_sesi11 import cari

load_dotenv()
client = genai.Client(api_key=os.environ["GEMINI_API_KEY"])

MODEL = "gemini-2.5-flash"
print("siap")

## 1. State: apa yang dibawa sistem dari awal sampai akhir

Semua node membaca dan menulis di satu tempat yang sama. Kalau sebuah informasi
tidak ada di State, node berikutnya tidak akan pernah melihatnya.

Perhatikan kunci `intent`. Ia tidak dipakai untuk menyusun jawaban. Ia ada
supaya kita bisa membuktikan, setelah sistem jalan, cabang mana yang tadi
dipilih. Sistem yang tidak bisa menjelaskan keputusannya sulit diperbaiki.

In [ ]:
class State(TypedDict):
    input: str
    intent: str
    konteks: str
    jawaban: str

## 2. Node pertama: intent detection

Inilah satu-satunya bagian yang benar-benar baru di 12.4. Sisanya adalah
perakitan dari komponen yang sudah Anda punya.

Node ini melakukan satu panggilan LLM yang hanya boleh mengembalikan satu kata.
Dua hal yang membuatnya bisa diandalkan:

1. Prompt-nya menyebut daftar label secara eksplisit, dan melarang penjelasan.
2. Hasilnya tetap dibersihkan dan diperiksa di Python. Model bisa saja
   mengembalikan sesuatu di luar daftar, dan sistem harus tetap berjalan.

Aturan nomor dua berlaku untuk setiap keputusan yang Anda serahkan ke LLM.
Percaya pada prompt, tetapi tetap pasang jaring pengaman.

In [ ]:
PROMPT_INTENT = """Klasifikasikan pertanyaan pengguna ke SATU label berikut:

data     - menanyakan isi dokumen kebijakan asuransi LindungiKu
tool     - meminta perhitungan angka, konversi, persentase, atau cicilan
langsung - obrolan umum yang tidak butuh dokumen maupun hitungan

Jawab dengan satu kata saja, tanpa penjelasan.

PERTANYAAN: {input}"""

SAH = {"data", "tool", "langsung"}


def deteksi_intent(state: State) -> dict:
    resp = client.models.generate_content(
        model=MODEL,
        contents=PROMPT_INTENT.format(input=state["input"]),
    )
    label = (resp.text or "").strip().lower().split()[0].strip(".,:")

    # TODO 1: pasang jaring pengaman. Apabila label yang dikembalikan model
    # tidak ada di dalam SAH, ganti menjadi "langsung" supaya sistem tidak
    # berhenti hanya karena satu jawaban model yang di luar dugaan.

    return {"intent": label}


print(deteksi_intent({"input": "Berapa masa tunggu klaim rawat inap?"}))
print(deteksi_intent({"input": "Berapa 15 persen dari 50 juta?"}))
print(deteksi_intent({"input": "Halo, kamu bisa bantu apa saja?"}))

## 3. Tiga cabang

Tidak ada satu pun konsep baru di bawah ini. Cabang `rag` memanggil retrieval
Sesi 11. Cabang `alat` memakai tool calling seperti di 12.3. Cabang `langsung`
tidak mengambil bahan apa pun.

Itulah inti modularitas: komponen yang dibangun terpisah masuk ke sini tanpa
diubah sedikit pun.

Perhatikan bahwa ketiganya menulis ke kunci State yang sama, yaitu `konteks`.
Karena bentuk keluarannya seragam, node `jawab` berikutnya tidak perlu tahu
cabang mana yang tadi jalan.

In [ ]:
def kalkulator(a: float, b: float, operator: str) -> str:
    """Hitung operasi aritmatika dua angka.

    Args:
        a: angka pertama
        b: angka kedua
        operator: salah satu dari "+", "-", "*", "/"
    """
    if operator == "/" and b == 0:
        return "tidak terdefinisi, pembagi nol"
    hasil = {"+": a + b, "-": a - b, "*": a * b,
             "/": a / b if b else 0.0}[operator]
    return str(hasil)


def rag(state: State) -> dict:
    # TODO 2: panggil cari() dengan state["input"], k=3, lalu gabungkan kunci
    # 'teks' dari setiap potongan menjadi satu blok, dan kembalikan sebagai
    # {"konteks": blok}.
    return {"konteks": ""}


def alat(state: State) -> dict:
    # TODO 3: panggil model dengan config=types.GenerateContentConfig(
    # tools=[kalkulator]) supaya model boleh memakai kalkulator sendiri, lalu
    # kembalikan teks jawabannya sebagai 'konteks'.
    return {"konteks": ""}


def langsung(state: State) -> dict:
    return {"konteks": ""}

## 4. Titik temu: satu node `jawab`

Ketiga cabang bertemu di sini. Node ini menyusun jawaban akhir dari `konteks`
apa pun yang tersedia.

Inilah yang disebut response aggregation. Tanpa titik temu, setiap cabang akan
mengeluarkan jawaban dengan gaya berbeda, dan sistem terasa seperti tiga
program yang ditempel, bukan satu asisten.

In [ ]:
PROMPT_JAWAB = """Anda adalah asisten layanan nasabah asuransi LindungiKu.
Jawab dengan ramah, ringkas, dan dalam Bahasa Indonesia.

Apabila KONTEKS di bawah terisi, gunakan hanya isi KONTEKS dan jangan menambah
informasi dari luar. Apabila KONTEKS kosong, jawab dari pengetahuan umum saja
dan jangan mengarang isi kebijakan.

KONTEKS:
{konteks}

PERTANYAAN: {input}"""


def jawab(state: State) -> dict:
    resp = client.models.generate_content(
        model=MODEL,
        contents=PROMPT_JAWAB.format(
            konteks=state["konteks"] or "(kosong)",
            input=state["input"],
        ),
    )
    return {"jawaban": (resp.text or "").strip()}

## 5. Merakit graph

Sekarang bagian yang menyambungkan semuanya. Bacalah seperti membaca diagram:
mulai, deteksi intent, lalu bercabang, lalu semuanya bertemu lagi di `jawab`.

`pilih_cabang` adalah fungsi biasa yang hanya membaca State dan mengembalikan
nama tujuan. Ia tidak memanggil LLM sama sekali. Keputusannya sudah dibuat di
node `deteksi_intent`, dan fungsi ini hanya menerjemahkannya menjadi arah.

In [ ]:
def pilih_cabang(state: State) -> str:
    return state["intent"]


b = StateGraph(State)

b.add_node("deteksi_intent", deteksi_intent)
b.add_node("rag", rag)
b.add_node("alat", alat)
b.add_node("langsung", langsung)
b.add_node("jawab", jawab)

b.add_edge(START, "deteksi_intent")

# TODO 4: sambungkan percabangannya.
# 1. add_conditional_edges dari "deteksi_intent", memakai pilih_cabang, dengan
#    peta tujuan {"data": "rag", "tool": "alat", "langsung": "langsung"}.
# 2. tiga add_edge, dari rag, alat, dan langsung, menuju "jawab".
# 3. satu add_edge dari "jawab" menuju END.

asisten = b.compile()
print(asisten.get_graph().draw_ascii())

## 6. Tiga skenario, tiga jalur

Satu fungsi pembungkus, supaya setiap kali sistem dijalankan kita melihat
cabang mana yang dipilih, bukan hanya jawabannya.

Jalankan ketiganya berurutan dan perhatikan baris INTENT. Pertanyaan yang
berbeda masuk lewat pintu yang sama dan keluar lewat jalan yang berbeda, tanpa
satu pun `if` yang kita tulis di tingkat aplikasi.

In [ ]:
def tanya(pertanyaan):
    hasil = asisten.invoke({
        "input": pertanyaan,
        "intent": "",
        "konteks": "",
        "jawaban": "",
    })
    print(f"PERTANYAAN : {pertanyaan}")
    print(f"INTENT     : {hasil['intent']}")
    print("JAWABAN    :")
    print(textwrap.indent(textwrap.fill(hasil["jawaban"], 66), "  "))
    print("-" * 74)
    return hasil


tanya("Berapa lama masa tunggu klaim rawat inap?")
print()
tanya("Berapa 15 persen dari 50 juta?")
print()
tanya("Halo, kamu bisa bantu apa saja?")

## 7. Kembali ke instruksi yang membuat 12.1 menyerah

Ingat instruksi yang mematahkan pipeline linear di awal sesi. Jalankan sekarang
lewat sistem yang baru.

Jawabannya belum tentu sempurna, dan itu wajar. Router mengarahkan satu
pertanyaan ke satu cabang, sedangkan instruksi tadi membutuhkan dua cabang
sekaligus. Yang berubah adalah sistemnya kini punya tempat untuk diperbaiki,
yaitu satu node pemecah instruksi dan satu edge tambahan. Pipeline linear tidak
punya tempat itu sama sekali.

In [ ]:
KOMPLEKS = (
    "Klaim rawat inap saya ditolak setelah menunggu 50 hari. Berapa masa tunggu resminya?"
    "Jika saya sudah menunggu lebih lama dari itu, hitung selisihnya dalam hari, lalu jelaskan prosedur bandingnya."
)

_ = tanya(KOMPLEKS)

## 8. Menambah kemampuan tanpa membongkar apa pun

Inilah ujian sesungguhnya dari sebuah sistem modular. Kita tambahkan satu
cabang baru, lalu kita hitung berapa banyak kode lama yang harus diubah.

Jawabannya: tidak satu pun. Kita hanya menambah satu node, satu label di
prompt, dan satu baris di peta tujuan.

Perhatikan apa yang **tidak** Anda sentuh: node `jawab`, ketiga cabang lama,
dan fungsi `tanya` semuanya tetap seperti semula.

In [ ]:
PROMPT_INTENT_BARU = PROMPT_INTENT.replace(
    "langsung - obrolan umum",
    "eskalasi - nasabah marah, mengancam, atau minta bicara dengan manusia\n"
    "langsung - obrolan umum",
)
SAH_BARU = SAH | {"eskalasi"}


def deteksi_intent_baru(state: State) -> dict:
    resp = client.models.generate_content(
        model=MODEL,
        contents=PROMPT_INTENT_BARU.format(input=state["input"]),
    )
    label = (resp.text or "").strip().lower().split()[0].strip(".,:")
    if label not in SAH_BARU:
        label = "langsung"
    return {"intent": label}


def eskalasi(state: State) -> dict:
    return {"konteks": "Sampaikan permohonan maaf dan beri tahu bahwa keluhan "
                       "diteruskan ke agen manusia dalam 1x24 jam."}


b2 = StateGraph(State)
b2.add_node("deteksi_intent", deteksi_intent_baru)
b2.add_node("rag", rag)
b2.add_node("alat", alat)
b2.add_node("langsung", langsung)
b2.add_node("eskalasi", eskalasi)
b2.add_node("jawab", jawab)

b2.add_edge(START, "deteksi_intent")
b2.add_conditional_edges("deteksi_intent", pilih_cabang, {
    "data": "rag",
    "tool": "alat",
    "langsung": "langsung",
    "eskalasi": "eskalasi",
})
for cabang in ("rag", "alat", "langsung", "eskalasi"):
    b2.add_edge(cabang, "jawab")
b2.add_edge("jawab", END)

asisten = b2.compile()
print(asisten.get_graph().draw_ascii())
_ = tanya("Klaim saya ditolak lagi. Saya mau bicara dengan orangnya langsung.")